In [ ]:
# INSTALL
!pip install -q openai-whisper transformers sentence-transformers faiss-cpu gTTS pydub soundfile librosa
!apt-get install ffmpeg -qq

In [ ]:
import whisper
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from gtts import gTTS
from pydub import AudioSegment
from google.colab import files
from IPython.display import Audio


# SPEECH TO TEXT (WHISPER)
class WhisperSTT:

    def __init__(self):
        print("Loading Whisper model...")
        self.model = whisper.load_model("medium")

    def speech_to_text(self, audio_path):
        result = self.model.transcribe(
            audio_path,
            task="translate",
            language="te",
            fp16=False
        )
        return str(result["text"]).strip()


# TRANSLATOR (NLLB)
class Translator:

    def __init__(self):

        print("Loading translation model...")

        model_name = "facebook/nllb-200-distilled-600M"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        self.lang_map = {
            "eng": "eng_Latn",
            "hin": "hin_Deva",
            "tel": "tel_Telu",
            "ben": "ben_Beng",
            "urd": "urd_Arab"
        }

    def translate(self, text, target_lang):

        tgt = self.lang_map.get(target_lang, "eng_Latn")

        inputs = self.tokenizer(text, return_tensors="pt")

        forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt)

        output = self.model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id
        )

        return self.tokenizer.batch_decode(output, skip_special_tokens=True)[0]


# RETRIEVER
class JinaRetriever:

    def __init__(self):

        print("Loading embedding model...")

        self.encoder = SentenceTransformer("intfloat/multilingual-e5-base")

        self.index = None
        self.docs = []

    def build_index(self, documents):

        prefixed_docs = ["passage: " + doc for doc in documents]

        embeddings = self.encoder.encode(
            prefixed_docs,
            convert_to_numpy=True
        )

        print("Embedding shape:", embeddings.shape)

        dimension = embeddings.shape[1]

        self.index = faiss.IndexFlatIP(dimension)

        faiss.normalize_L2(embeddings)

        self.index.add(embeddings)

        self.docs = [{"text": doc} for doc in documents]

    def retrieve(self, query, top_k=3):

        query = "query: " + str(query)

        query_emb = self.encoder.encode(
            [query],
            convert_to_numpy=True
        )

        faiss.normalize_L2(query_emb)

        scores, indices = self.index.search(query_emb, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            results.append({
                "text": self.docs[idx]["text"],
                "score": float(score)
            })

        return results


# TEXT TO SPEECH
class LocalTTS:

    def synthesize(self, text, lang="en", output="answer.mp3"):

        tts = gTTS(text=text, lang=lang)

        tts.save(output)

        return output


# PIPELINE
class IndicQAPipeline:

    def __init__(self, stt_model, retriever, translator, tts_model, lang_map):

        self.stt = stt_model
        self.retriever = retriever
        self.translator = translator
        self.tts = tts_model
        self.lang_map = lang_map

        # scheme keywords for relevance detection
        self.scheme_keywords = [
            "bharati", "kisan", "ayushman", "rythu", "bima",
            "scheme", "yojana", "farmer", "Ayushman Bharat "
        ]


    def is_relevant(self, query, score):
        query_lower = query.lower()
        keyword_match = any(k in query_lower for k in self.scheme_keywords)

        if keyword_match:
            # Queries that mention a scheme keyword → use a lower threshold
            return score >= 0.55
        else:
            # Queries without any keyword → need very high confidence
            return score >= 0.8   # adjust this value as needed


    def select_answer(self, query, retrieved):
        if not retrieved:
            return "Please ask a question related to government schemes."
        top_score = retrieved[0]["score"]
        if not self.is_relevant(query, top_score):
            return "Please ask a question related to government schemes."

        # Multi-answer logic (unchanged)
        MULTI_MARGIN = 0.05
        filtered = [r for r in retrieved if r["score"] >= top_score - MULTI_MARGIN]

        if len(filtered) > 1:
            return " ".join(r["text"] for r in filtered[:2])
        return retrieved[0]["text"]


    def process_text(self, query, user_lang="eng", return_speech=True):
        retrieved = self.retriever.retrieve(query)
        answer_en = self.select_answer(query, retrieved)   # <-- pass query
        answer = self.translator.translate(answer_en, user_lang)
        if return_speech:
            lang = self.lang_map.get(user_lang, "en")
            audio = self.tts.synthesize(answer, lang)
            return answer, audio
        return answer

    def process_speech(self, audio_path, user_lang="eng", return_speech=True):
        query = self.stt.speech_to_text(audio_path)
        print("Recognized text:", query)
        retrieved = self.retriever.retrieve(query)
        answer_en = self.select_answer(query, retrieved)   # <-- pass query here
        answer = self.translator.translate(answer_en, user_lang)
        if return_speech:
            lang = self.lang_map.get(user_lang, "en")
            audio = self.tts.synthesize(answer, lang)
            return answer, audio
        return answer

# YOUR DATASET

schemes_data = [
    {
        "name": "Bharati Scheme for Education",
        "sections": [
            "Bharati Scheme for Education – Description: A financial assistance program for Brahmin students in Andhra Pradesh studying from 1st to 5th class.",
            "Bharati Scheme for Education – Benefits: Eligible students receive ₹5,000 financial assistance per academic year.",
            "Bharati Scheme for Education – Eligibility: Students must belong to the Brahmin community and study in Andhra Pradesh. Family income should not exceed ₹3,00,000 per year.",
            "Bharati Scheme for Education – Priority Categories: Orphans, differently-abled students, and children of single parents are given priority.",
            "Bharati Scheme for Education – Documents Required: Aadhaar card of student and parents, caste certificate, birth certificate, study certificate, and bank passbook.",
            "Bharati Scheme for Education – Where to Apply: Applications must be submitted through the Andhra Pradesh Brahmin Welfare Corporation official portal.",
            "Bharati Scheme for Education – Application Process: Register on the scheme portal, fill the application form with personal details, upload documents, and submit the application.",
            "Bharati Scheme for Education – Selection Authority: Beneficiaries are selected by the State Level Selection Committee of the AP Brahmin Welfare Corporation.",
            "Bharati Scheme for Education – Application Status: Applicants can check status online using Aadhaar number, reference number, or mobile number.",
            "Bharati Scheme for Education – Acknowledgement Slip: Applicants can download the acknowledgement slip from the scheme website after submission."
        ]
    },
    {
        "name": "PM-Kisan Samman Nidhi",
        "sections": [
            "PM-Kisan Samman Nidhi – Description: A central government scheme providing financial support to small and marginal farmers.",
            "PM-Kisan Samman Nidhi – Benefits: Eligible farmers receive ₹6,000 per year in three equal installments directly into their bank accounts.",
            "PM-Kisan Samman Nidhi – Eligibility: Small and marginal farmers owning cultivable land are eligible.",
            "PM-Kisan Samman Nidhi – Documents Required: Aadhaar card, bank account details, and land ownership records.",
            "PM-Kisan Samman Nidhi – Where to Apply: Farmers can apply online through the PM-Kisan portal or via local agriculture offices.",
            "PM-Kisan Samman Nidhi – Payment Method: Payments are transferred directly to beneficiaries through Direct Benefit Transfer (DBT).",
            "PM-Kisan Samman Nidhi – Status Check: Farmers can check payment status using Aadhaar number or registration ID on the official portal."
        ]
    },
    {
        "name": "Ayushman Bharat",
        "sections": [
            "Ayushman Bharat – Description: A national health protection scheme providing health insurance coverage for economically vulnerable families.",
            "Ayushman Bharat – Benefits: Health insurance coverage up to ₹5,00,000 per family per year.",
            "Ayushman Bharat – Eligibility: Families identified in the SECC database are eligible.",
            "Ayushman Bharat – Documents Required: Aadhaar card, ration card, and family identification proof.",
            "Ayushman Bharat – Where to Use: Beneficiaries can receive treatment at empanelled public or private hospitals.",
            "Ayushman Bharat – Registration: Eligible families can register through empanelled hospitals or the official website.",
            "Ayushman Bharat – Services Covered: The scheme covers hospitalization, surgery, medicines, and diagnostic services."
        ]
    },
    {
        "name": "Rythu Bima Scheme",
        "sections": [
            "Rythu Bima Scheme – Description: A Telangana government life insurance scheme for farmers.",
            "Rythu Bima Scheme – Benefits: Insurance coverage of ₹5,00,000 is provided to the nominee in case of the farmer's death.",
            "Rythu Bima Scheme – Eligibility: Farmers aged between 18 and 59 years registered with the agriculture department are eligible.",
            "Rythu Bima Scheme – Documents Required: Aadhaar card, land ownership proof, and bank account details.",
            "Rythu Bima Scheme – Implementation: The scheme is implemented through LIC and the Telangana agriculture department.",
            "Rythu Bima Scheme – Claim Process: Nominees submit required documents to receive insurance compensation."
        ]
    }
]


# BUILD DOCUMENTS

documents = []

for scheme in schemes_data:
    documents.extend(scheme["sections"])



# INITIALIZE SYSTEM

stt_model = WhisperSTT()

translator = Translator()

retriever = JinaRetriever()

retriever.build_index(documents)

tts_model = LocalTTS()

lang_map = {
    "eng": "en",
    "hin": "hi",
    "tel": "te",
    "ben": "bn",
    "urd": "ur"
}

pipeline = IndicQAPipeline(
    stt_model,
    retriever,
    translator,
    tts_model,
    lang_map
)

print("System Ready")

In [ ]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]

audio = AudioSegment.from_file(filename)

audio = audio.set_frame_rate(16000).set_channels(1)

audio.export("input.wav", format="wav")

answer, audio = pipeline.process_speech(
    "input.wav",
    user_lang="tel"   #TO DETECT TELUGU LANGUAGE
)

print(answer)

display(Audio(audio))

In [ ]:
answer, audio = pipeline.process_text(
        "who can apply rythu bima and its benefits?",
        user_lang="eng",
        return_speech=True
    )

print("Answer:", answer)

In [ ]:
questions = [
    ("hin", "आयुष्मान भारत योजना में कितना बीमा कवर मिलता है?"),
    ("tel", "భారతి పథకం కోసం ఎక్కడ దరఖాస్తు చేసుకోవాలి?"),
    ("urd", "پردھان منتری کسان سمن ندھی اسکیم کے فوائد کیا ہیں؟"),
    ("ben", "আয়ুষ্মান ভারত প্রকল্পের জন্য কারা যোগ্য?"),
    ("eng", "What documents are required for Rythu Bima?")
]

for lang, q in questions:

    print(f"\n[{lang}] Question: {q}")

    answer, audio = pipeline.process_text(
        q,
        user_lang=lang,
        return_speech=True
    )

    print("Answer:", answer)

    display(Audio(audio))

In [ ]:
answer, audio = pipeline.process_text(
        "what is ai?",
        user_lang="eng",
        return_speech=True
    )

print("Answer:", answer)

In [ ]:
test_set_multilingual = [
    # Hindi
    {
        "lang": "hin",
        "question": "प्रधानमंत्री किसान सम्मान निधि योजना में लाभ की राशि कितनी है?",
        "expected_answer": "पीएम-किसान सम्मान निधि  लाभ: पात्र किसानों को ₹6,000 प्रति वर्ष तीन समान किश्तों में सीधे उनके बैंक खातों में मिलते हैं।"
    },
    {
        "lang": "hin",
        "question": "आयुष्मान भारत योजना के लिए कौन पात्र है?",
        "expected_answer": "आयुष्मान भारत  पात्रता: एसईसीसी डेटाबेस में पहचाने गए परिवार पात्र हैं।"
    },
    {
        "lang": "hin",
        "question": "भारती शिक्षा योजना के लिए आय सीमा क्या है?",
        "expected_answer": "भारती शिक्षा योजना  पात्रता: विद्यार्थी ब्राह्मण समुदाय से होने चाहिए और आंध्र प्रदेश में पढ़ रहे हों। परिवार की वार्षिक आय ₹3,00,000 से अधिक नहीं होनी चाहिए।"
    },
    # Telugu
    {
        "lang": "tel",
        "question": "భారతి విద్యా పథకం కింద లాభం మొత్తం ఎంత?",
        "expected_answer": "భారతి విద్యా పథకం  ప్రయోజనాలు: అర్హులైన విద్యార్థులకు ప్రతి విద్యా సంవత్సరానికి ₹5,000 ఆర్థిక సహాయం."
    },
    {
        "lang": "tel",
        "question": "పీఎం-కిసాన్ సమ్మాన్ నిధి పథకానికి ఎవరు అర్హులు?",
        "expected_answer": "పీఎం-కిసాన్ సమ్మాన్ నిధి  అర్హత: సాగు భూమి కలిగిన చిన్న మరియు సన్నకారు రైతులు అర్హులు."
    },
    {
        "lang": "tel",
        "question": "ఆయుష్మాన్ భారత్ పథకం ప్రయోజనాలు ఏమిటి?",
        "expected_answer": "ఆయుష్మాన్ భారత్  ప్రయోజనాలు: కుటుంబానికి సంవత్సరానికి ₹5,00,000 వరకు ఆరోగ్య బీమా కవరేజీ."
    },

    # Bengali
    {
        "lang": "ben",
        "question": "রাইথু বিমা প্রকল্পের জন্য কী কী নথি প্রয়োজন?",
        "expected_answer": "রাইথু বিমা প্রকল্প – প্রয়োজনীয় নথি: আধার কার্ড, জমির মালিকানার প্রমাণ, এবং ব্যাংক অ্যাকাউন্টের বিবরণ।"
    },
    {
        "lang": "ben",
        "question": "পিএম-কিসান সম্মান নিধি প্রকল্পে কারা যোগ্য?",
        "expected_answer": "পিএম-কিসান সম্মান নিধি  যোগ্যতা: ক্ষুদ্র ও প্রান্তিক কৃষক যাদের চাষযোগ্য জমি আছে, তারা যোগ্য।"
    },
    {
        "lang": "ben",
        "question": "আয়ুষ্মান ভারত প্রকল্পের আওতায় কত বীমা কভার পাওয়া যায়?",
        "expected_answer": "আয়ুষ্মান ভারত  সুবিধা: পরিবার প্রতি বছরে ₹5,00,000 পর্যন্ত স্বাস্থ্য বীমা কভারেজ।"
    },
    # Urdu
    {
        "lang": "urd",
        "question": "ریتھو بیما اسکیم کے لیے کون سے دستاویزات درکار ہیں؟",
        "expected_answer": "ریتھو بیما اسکیم – مطلوبہ دستاویزات: آدھار کارڈ، زمین کی ملکیت کا ثبوت، اور بینک اکاؤنٹ کی تفصیلات۔"
    },
    {
        "lang": "urd",
        "question": "بھارتی تعلیم اسکیم کے تحت فائدہ کی رقم کتنی ہے؟",
        "expected_answer": "بھارتی تعلیم اسکیم – فوائد: اہل طلباء کو تعلیمی سال کے دوران ₹5,000 کی مالی امداد دی جاتی ہے۔"
    },
    {
        "lang": "urd",
        "question": "آیوشمان بھارت اسکیم کے فوائد کیا ہیں؟",
        "expected_answer": "آیوشمان بھارت – فوائد: فی خاندان سالانہ ₹5,00,000 تک کی صحت بیمہ کوریج۔"
    },
    # English (using dash format)
    {
        "lang": "eng",
        "question": "What is the benefit amount under Bharati Scheme for Education?",
        "expected_answer": "Bharati Scheme for Education  Benefits: Eligible students receive ₹5,000 financial assistance per academic year."
    },
    {
        "lang": "eng",
        "question": "What documents are needed for PM-Kisan Samman Nidhi?",
        "expected_answer": "PM-Kisan Samman Nidhi  Documents Required: Aadhaar card, bank account details, and land ownership records."
    },
    {
        "lang": "eng",
        "question": "How to claim Rythu Bima?",
        "expected_answer": "Rythu Bima Scheme  Claim Process: Nominees submit required documents to receive insurance compensation."
    },
]

In [ ]:
!pip install rouge-score nltk

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np
from sentence_transformers import SentenceTransformer

nltk.download('punkt')

embedder = SentenceTransformer("intfloat/multilingual-e5-base")

def tokenize(text):
    return text.split()

def compute_metrics(predicted, reference):
    pred_tokens = tokenize(predicted)
    ref_tokens_bleu = [tokenize(reference)]
    ref_tokens_f1 = tokenize(reference)

    # BLEU
    smooth = SmoothingFunction().method1
    bleu = sentence_bleu(ref_tokens_bleu, pred_tokens, smoothing_function=smooth)

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    scores = scorer.score(reference, predicted)
    rouge1_f1 = scores['rouge1'].fmeasure
    rougeL_f1 = scores['rougeL'].fmeasure

    # Token F1
    pred_set = set(pred_tokens)
    ref_set = set(ref_tokens_f1)
    if len(pred_set) == 0 and len(ref_set) == 0:
        token_f1 = 1.0
    else:
        precision = len(pred_set & ref_set) / len(pred_set) if len(pred_set) > 0 else 0
        recall = len(pred_set & ref_set) / len(ref_set) if len(ref_set) > 0 else 0
        token_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Return a DICTIONARY, not a set
    return {
        "bleu": bleu,
        "rouge1_f1": rouge1_f1,
        "rougeL_f1": rougeL_f1,
        "token_f1": token_f1
    }

def semantic_similarity(pred, ref):
    pred_emb = embedder.encode(["query: " + pred])[0]
    ref_emb = embedder.encode(["query: " + ref])[0]
    return np.dot(pred_emb, ref_emb) / (np.linalg.norm(pred_emb) * np.linalg.norm(ref_emb))

results = []
for test in test_set_multilingual:
    lang = test["lang"]
    question = test["question"]
    expected = test["expected_answer"]

    pred_answer = pipeline.process_text(question, user_lang=lang, return_speech=False)

    metrics = compute_metrics(pred_answer, expected)
    metrics["sem_sim"] = semantic_similarity(pred_answer, expected)
    metrics["lang"] = lang
    metrics["question"] = question
    results.append(metrics)

# Print results
for r in results:
    print(f"\n[{r['lang']}] Q: {r['question']}")
    print(f"BLEU: {r['bleu']:.4f}, ROUGE-1 F1: {r['rouge1_f1']:.4f}, SemSim: {r['sem_sim']:.4f}")

# Averages
print(f"\nOverall: BLEU={np.mean([r['bleu'] for r in results]):.4f}, ROUGE-1 F1={np.mean([r['rouge1_f1'] for r in results]):.4f}, SemSim={np.mean([r['sem_sim'] for r in results]):.4f}")